In [ ]:
import sqlite3
import pickle
import array
import os
from tqdm import tqdm

# Create a SQLite database
conn = sqlite3.connect('knowledge.db', timeout=100)
conn.execute('BEGIN')
conn.execute('''CREATE TABLE IF NOT EXISTS clauses (id INTEGER PRIMARY KEY, token INTEGER, weight INTEGER, literals BLOB)''')

user_home = os.path.expanduser("~")
knowledge = os.path.join(user_home, "knowledge")
# Loop over all files in the directory
id_counter = 0  # Counter for id values specific to each file

files_progress_bar = tqdm(total=len(os.listdir(knowledge)), desc="Running Files")
for filename in os.listdir(knowledge):
    if filename.endswith('.pkl'):  # Only consider pickle files
        file_path = os.path.join(knowledge, filename)
        token = int(os.path.splitext(filename)[0])  # Extract token value from filename
        with open(file_path, 'rb') as f:
            data = pickle.load(f)           
            with conn:  # Use a context manager for the transaction
                for weight, literals in data:
                    weight = int(weight) 
                    literals_array = array.array('h', literals)  # Convert literals list to signed short integer array
                    literals_bytes = literals_array.tobytes()  # Convert byte array to bytes
                    conn.execute('''INSERT INTO clauses (id, token, weight, literals) VALUES (?, ?, ?, ?)''', (id_counter, token, weight, literals_bytes))
                    id_counter += 1
    files_progress_bar.update(1)
files_progress_bar.close()
conn.close()

In [4]:
import sqlite3
import array

conn = sqlite3.connect('knowledge.db')
c = conn.cursor()
token = 13788
c.execute('''SELECT weight, literals FROM clauses WHERE token = ? AND weight > 0''', (token,))
rows = c.fetchall()

weights = []
literals_list = []

for row in rows:
    weight, literals_bytes = row
    literals_array = array.array('h')
    literals_array.frombytes(literals_bytes)  # Convert bytes to byte array
    literals = list(literals_array)  # Convert byte array to list
    data = (weight, literals)
    print(f'Data: {data}')

conn.close()

Data: (13, [8881])
Data: (11, [5338, 10306, 13130])
Data: (17, [6154, 13470])
Data: (4, [1932, 4221, 4642, 10470, 16736, 17365])
Data: (3, [7894])
Data: (4, [1376])
Data: (14, [3641, 5012, 5270, 7405, 15217])
Data: (12, [6800, 12332, 16179, 18504])
Data: (18, [11004, 18092])
Data: (16, [1171, 3489, 6901, 14656])
Data: (4, [5338, 7136, 12332])
Data: (1, [618, 6557, 12164, 19387])
Data: (19, [433, 13433, 14867, 19679])
Data: (6, [6346, 15084])
Data: (16, [1130, 4217, 6574, 12329])
Data: (22, [1834, 6346, 16187])
Data: (5, [16197, 16878])
Data: (4, [16505, 17409])
Data: (27, [4744, 8053, 10727, 15846])
Data: (24, [672, 6593, 7720])
Data: (14, [241, 1247, 5270])
Data: (24, [29, 3190, 14260, 14265, 17967, 19476])
Data: (17, [4108, 6141, 17267])
Data: (26, [9732, 17967])
Data: (16, [3846, 8098, 17134])
Data: (14, [247, 1130, 16031, 19293])
Data: (4, [4643])
Data: (7, [15635])
Data: (2, [2757, 18629])
Data: (9, [13434, 16736])
Data: (15, [674, 4677, 13436, 18092, 19532, 19930])
Data: (18, [55

In [4]:
import os
user_home = os.path.expanduser("~")
knowledge = os.path.join(user_home, "knowledge")

for filename in os.listdir(knowledge):
    if filename == "5.pkl":  # Only consider pickle files
        file_path = os.path.join(knowledge, filename)
        token = int(os.path.splitext(filename)[0])  # Extract token value from filename
        with open(file_path, 'rb') as f:
            data = pickle.load(f)  
            print(data) 

[[-18, [12512]], [-1, [19501]], [-22, [3857, 16171, 19737]], [3, [7061, 11214, 13718]], [23, [4208, 5581, 7295, 10297, 16045, 17227]], [4, [8631, 13106, 14641]], [-10, [15232]], [12, [4932, 17460]], [-12, [16887]], [16, [3676, 4423, 13854, 16468]], [16, [7780, 9024, 18911]], [31, [4908, 11707, 13747]], [12, [460, 11128]], [0, [12518]], [22, [241, 8959, 19707]], [-23, [4202]], [-27, [10915, 17270]], [21, [5127, 19401]], [-16, [233, 6495, 8916, 16693]], [3, [8192, 18741]], [-24, [11366, 13930, 19919]], [-17, [9336, 10816]], [-3, [3876]], [27, [8285, 13068, 14948, 15708, 16045, 19704]], [-1, [12164]], [-33, [10815, 13930]], [-25, [9420, 10883, 18023, 19737]], [-25, [6403, 13518, 15896]], [-17, [7913, 12001, 12545, 13995]], [-26, [2549, 9082, 15217]], [28, [3588, 4459, 15832]], [17, [1029, 3718, 15277, 17142]], [21, [4782, 6694, 10413, 11125, 14156]], [-28, [6031, 18900, 19930]], [-9, [2021]], [13, [2457, 10047, 17479, 18428]], [-2, [3941, 6508, 8296]], [-36, [6036, 13191]], [28, [234, 945

In [ ]:
import sqlite3

# Create a SQLite database connection and cursor
conn = sqlite3.connect('knowledge.db')
c = conn.cursor()

# Delete all records from the clauses table
c.execute('DELETE FROM clauses')

# Commit changes and close database connection
conn.commit()
conn.close()

In [ ]:
conn.close()

In [ ]:
import sqlite3
import psycopg2
import array

# Connect to SQLite database
sqlite_conn = sqlite3.connect('knowledge.db')
sqlite_cursor = sqlite_conn.cursor()

# Connect to PostgreSQL database
pg_conn = psycopg2.connect(
    dbname='your_postgresql_db',
    user='your_username',
    password='your_password',
    host='your_host',
    port='your_port'
)
pg_cursor = pg_conn.cursor()

# Select data from SQLite
token = 13788
sqlite_cursor.execute("SELECT weight, literals FROM clauses WHERE token = ? AND weight > 0", (token,))
rows = sqlite_cursor.fetchall()

# Transfer data to PostgreSQL
for row in rows:
    weight, literals_bytes = row
    literals_array = array.array('h')
    literals_array.frombytes(literals_bytes)
    literals = list(literals_array)
    data = (weight, literals)
    pg_cursor.execute("INSERT INTO clauses (weight, literals) VALUES (%s, %s)", data)

# Commit changes and close connections
pg_conn.commit()
pg_cursor.close()
pg_conn.close()
sqlite_cursor.close()
sqlite_conn.close()

In [2]:
import psycopg2

# Connect to the PostgreSQL server
conn = psycopg2.connect(
    dbname='knowledge.pstgrs',
    user='cair',
    password='cair',
    host='localhost',
    port='15432'
)

# Create a cursor object to execute SQL queries
cursor = conn.cursor()

# Create a new database
cursor.execute('CREATE DATABASE knowledge;')

# Connect to the new database
conn = psycopg2.connect(
    dbname='knowledge',
    user='cair',
    password='cair',
    host='localhost',
    port='15432'
)

# Create a new table
cursor.execute('''
    CREATE TABLE clauses (
        id SERIAL PRIMARY KEY,
        token INTEGER NOT NULL,
        weight INTEGER NOT NULL,
        literals INTEGER[] NOT NULL
    );
''')

# Commit changes and close connection
conn.commit()
cursor.close()
conn.close()


OperationalError: connection to server at "localhost" (::1), port 15432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 15432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
